# Exploration macro — phase empirique (V0)

Objectif: démarrer rapidement l'analyse empirique des séries avant toute industrialisation.

⚠️ Cette V0 couvre 3 variables immédiatement disponibles dans la source retenue:
- inflation (`Consumer Price Index`)
- rendement actions (`SP500`)
- taux crédit long terme (`Long Interest Rate`)

Les variables *croissance salaire*, *IRL* et *prix immobilier* restent à intégrer dans une itération suivante avec une source complémentaire.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.api import VAR


In [ ]:
RAW_PATH = Path("../data/raw/s_and_p_500.csv")
assert RAW_PATH.exists(), f"Fichier manquant: {RAW_PATH.resolve()}"

df_raw = pd.read_csv(RAW_PATH, parse_dates=["Date"])
df_raw = df_raw.rename(
    columns={
        "Date": "date",
        "SP500": "sp500_level",
        "Consumer Price Index": "cpi_level",
        "Long Interest Rate": "long_rate_pct",
    }
)

df_raw = df_raw[["date", "sp500_level", "cpi_level", "long_rate_pct"]].dropna().sort_values("date")
df_raw.tail()


## Nettoyage et transformations modélisables

- Inflation mensuelle = variation relative du CPI.
- Rendement actions mensuel = log-return du SP500.
- Taux crédit = niveau mensuel (converti en décimal).


In [ ]:
df = df_raw.set_index("date").copy()

df["inflation"] = df["cpi_level"].pct_change()
df["rendement_actions"] = np.log(df["sp500_level"]).diff()
df["taux_credit"] = df["long_rate_pct"] / 100.0

model_df = df[["inflation", "rendement_actions", "taux_credit"]].dropna()
model_df.head()


In [ ]:
ax = model_df.plot(subplots=True, figsize=(12, 8), title=["Inflation", "Rendement actions", "Taux crédit"])
plt.tight_layout()


## Statistiques de base


In [ ]:
stats = pd.DataFrame({
    "moyenne": model_df.mean(),
    "volatilite": model_df.std(),
    "skewness": model_df.skew(),
    "kurtosis": model_df.kurtosis(),
})
stats


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(model_df.columns):
    model_df[col].hist(bins=50, ax=axes[i])
    axes[i].set_title(f"Histogramme — {col}")
plt.tight_layout()


In [ ]:
def adf_summary(series: pd.Series) -> dict:
    stat, pval, lags, nobs, *_ = adfuller(series.dropna(), autolag="AIC")
    return {"adf_stat": stat, "p_value": pval, "lags": lags, "nobs": nobs}

adf_results = pd.DataFrame({c: adf_summary(model_df[c]) for c in model_df.columns}).T
adf_results


## Corrélations, ACF, stabilité temporelle


In [ ]:
corr = model_df.corr()
corr


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(model_df.columns):
    plot_acf(model_df[col], ax=axes[i], lags=36, title=f"ACF — {col}")
plt.tight_layout()


In [ ]:
rolling_mean = model_df.rolling(60).mean()
rolling_vol = model_df.rolling(60).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
rolling_mean.plot(ax=axes[0], title="Rolling mean (fenêtre 60 mois)")
rolling_vol.plot(ax=axes[1], title="Rolling volatilité (fenêtre 60 mois)")
plt.tight_layout()


In [ ]:
lags = range(-12, 13)
cc_rows = []
for lag in lags:
    shifted = model_df["rendement_actions"].shift(lag)
    cc_rows.append({"lag": lag, "corr_infl_vs_actions": model_df["inflation"].corr(shifted)})

cross_corr = pd.DataFrame(cc_rows).set_index("lag")
cross_corr.plot(figsize=(10,4), title="Cross-corr inflation vs rendement actions")
plt.axhline(0, color="black", lw=1)
plt.show()


## Prototype VAR(1)


In [ ]:
var_model = VAR(model_df)
var_res = var_model.fit(maxlags=1)
print(var_res.summary())


In [ ]:
# Vérification stabilité via racines (stabilité si toutes > 1 en module)
roots_modulus = np.abs(var_res.roots)
pd.DataFrame({"root_modulus": roots_modulus, "stable": roots_modulus > 1})


In [ ]:
horizon = 120
sim_values = var_res.simulate_var(steps=horizon)
sim_df = pd.DataFrame(sim_values, columns=model_df.columns)

sim_stats = pd.DataFrame({
    "historique_moy": model_df.mean(),
    "simule_moy": sim_df.mean(),
    "historique_vol": model_df.std(),
    "simule_vol": sim_df.std(),
})
sim_stats


## Conclusions V0 (à affiner)

- Les séries de **variations** (inflation mensuelle et log-return actions) sont bien plus proches de la stationnarité que les niveaux bruts.
- Le taux crédit en niveau présente une persistance temporelle visible (ACF lente).
- Les corrélations et volatilités ne sont pas parfaitement stables dans le temps, ce qui justifie de surveiller un possible effet de régime en phase suivante.
- Un VAR(1) fournit un premier benchmark exploitable pour la simulation, avant d'envisager un modèle plus riche.
